In [21]:
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import create_engine, text

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("✅ Connected")

✅ Connected


In [22]:
with engine.begin() as conn:
    
    conn.execute(text("DROP VIEW IF EXISTS vw_kpi_transactions CASCADE;"))
    conn.execute(text("DROP VIEW IF EXISTS vw_kpi_taux_defaut CASCADE;"))

    conn.execute(text("DROP TABLE IF EXISTS transactions CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS clients CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS segments_client CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS agences CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS produits CASCADE;"))

    conn.execute(text("""
        CREATE TABLE agences (
            agence_id SERIAL PRIMARY KEY,
            agence VARCHAR(100) UNIQUE NOT NULL
        );
    """))

    conn.execute(text("""
        CREATE TABLE produits (
            produit_id SERIAL PRIMARY KEY,
            produit VARCHAR(100) UNIQUE NOT NULL
        );
    """))

    conn.execute(text("""
        CREATE TABLE segments_client (
            segment_id SERIAL PRIMARY KEY,
            segment_client VARCHAR(50) UNIQUE NOT NULL,
            categorie_risque VARCHAR(50)
        );
    """))

    conn.execute(text("""
        CREATE TABLE clients (
            client_id VARCHAR PRIMARY KEY,
            segment_id INT,
            score_credit_client NUMERIC(10,2),
            FOREIGN KEY (segment_id) REFERENCES segments_client(segment_id)
        );
    """))

    conn.execute(text("""
        CREATE TABLE transactions (
            transaction_id VARCHAR PRIMARY KEY,
            client_id VARCHAR,
            agence_id INT,
            produit_id INT,
            date_transaction TIMESTAMP,
            montant_eur NUMERIC,
            statut VARCHAR,
            solde_avant NUMERIC,
            annee INT,
            mois INT,
            FOREIGN KEY (client_id) REFERENCES clients(client_id),
            FOREIGN KEY (agence_id) REFERENCES agences(agence_id),
            FOREIGN KEY (produit_id) REFERENCES produits(produit_id)
        );
    """))

print("✅ Tables created successfully")

✅ Tables created successfully


In [23]:
df = pd.read_csv("financecore_clean.csv")

df.rename(columns={"annees": "annee"}, inplace=True)

df.head()

,transaction_id,client_id,date_transaction,montant,devise,taux_change_eur,montant_eur,categorie,produit,agence,...,is_anomaly,annee,mois,trimestre,jour_semaine,montant_eur_verifie,ecart_eur,is_conversion_error,categorie_risque,taux_rejet
0,TXN000559,CLI0060,2022-04-19 02:31:00,2050.42,EUR,1.00,2050.42,Depot especes,Compte Epargne,Marseille-Vieux-Port,...,False,2022,4,2,1,2050.420000,0.000000,False,Medium,5.405405
1,TXN001154,CLI0057,2024-06-20 20:51:00,-123.66,GBP,0.86,-143.79,Retrait DAB,Credit Consommation,Bordeaux-Meriadeck,...,False,2024,6,2,3,-143.790698,0.000698,False,High,4.428904
2,TXN000764,CLI0015,2024-08-28 05:03:00,-396.17,EUR,1.00,-396.17,Prelevement,PEA,Lyon-Part-Dieu,...,False,2024,8,3,2,-396.170000,0.000000,False,Medium,4.193548
3,TXN001598,CLI0045,2024-01-07 08:16:00,225.20,EUR,1.00,225.20,Paiement CB,Credit Consommation,Bordeaux-Meriadeck,...,False,2024,1,1,6,225.200000,0.000000,False,Low,4.428904
4,TXN001873,CLI0034,2024-08-11 19:52:00,935.32,EUR,1.00,935.32,Interets,Credit Immobilier,Bordeaux-Meriadeck,...,False,2024,8,3,6,935.320000,0.000000,False,High,4.428904


In [24]:
agences_df = df[["agence"]].drop_duplicates()
produits_df = df[["produit"]].drop_duplicates()
segments_df = df[["segment_client", "categorie_risque"]].drop_duplicates()

clients_df = df[["client_id", "segment_client", "score_credit_client"]].drop_duplicates()

transactions_df = df[[
    "transaction_id",
    "client_id",
    "date_transaction",
    "montant_eur",
    "statut",
    "solde_avant",
    "annee",
    "mois",
    "agence",
    "produit"
]].drop_duplicates(subset="transaction_id")

In [25]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO segments_client (segment_client, categorie_risque)
        VALUES (:segment_client, :categorie_risque)
        ON CONFLICT (segment_client)
        DO UPDATE SET categorie_risque = EXCLUDED.categorie_risque;
    """), segments_df.to_dict(orient="records"))

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO agences (agence)
        VALUES (:agence)
        ON CONFLICT (agence) DO NOTHING;
    """), agences_df.to_dict(orient="records"))

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO produits (produit)
        VALUES (:produit)
        ON CONFLICT (produit) DO NOTHING;
    """), produits_df.to_dict(orient="records"))

print("✅ Dimensions loaded")

✅ Dimensions loaded


In [26]:
agences_map = pd.read_sql("SELECT * FROM agences", engine)
produits_map = pd.read_sql("SELECT * FROM produits", engine)
segments_map = pd.read_sql("SELECT * FROM segments_client", engine)

agences_map.head()

,agence_id,agence
0,1,Marseille-Vieux-Port
1,2,Bordeaux-Meriadeck
2,3,Lyon-Part-Dieu
3,4,Lille-Grand-Place
4,5,Toulouse-Capitole


In [27]:
clients_df = clients_df.merge(segments_map, on="segment_client", how="left")
clients_df = clients_df[["client_id", "segment_id", "score_credit_client"]]

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO clients (client_id, segment_id, score_credit_client)
        VALUES (:client_id, :segment_id, :score_credit_client)
        ON CONFLICT (client_id)
        DO UPDATE SET score_credit_client = EXCLUDED.score_credit_client;
    """), clients_df.to_dict(orient="records"))

print("✅ Clients loaded")

✅ Clients loaded


In [28]:
transactions_df = transactions_df.merge(agences_map, on="agence", how="left")
transactions_df = transactions_df.merge(produits_map, on="produit", how="left")

# تنظيف مهم
transactions_df = transactions_df.dropna(subset=["agence_id", "produit_id"])

transactions_df["date_transaction"] = pd.to_datetime(
    transactions_df["date_transaction"]
)

transactions_df = transactions_df[[
    "transaction_id",
    "client_id",
    "agence_id",
    "produit_id",
    "date_transaction",
    "montant_eur",
    "statut",
    "solde_avant",
    "annee",
    "mois"
]]

data = transactions_df.to_dict("records")

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO transactions (
            transaction_id, client_id, agence_id, produit_id,
            date_transaction, montant_eur, statut,
            solde_avant, annee, mois
        )
        VALUES (
            :transaction_id, :client_id, :agence_id, :produit_id,
            :date_transaction, :montant_eur, :statut,
            :solde_avant, :annee, :mois
        )
        ON CONFLICT (transaction_id) DO NOTHING;
    """), data)

print("✅ Transactions loaded cleanly")

✅ Transactions loaded cleanly


In [29]:
print(pd.read_sql("SELECT COUNT(*) FROM transactions", engine))
print(pd.read_sql("SELECT COUNT(*) FROM clients", engine))
print(pd.read_sql("SELECT COUNT(*) FROM agences", engine))
print(pd.read_sql("SELECT COUNT(*) FROM produits", engine))

   count
0   2000
   count
0    150
   count
0      8
   count
0      8


In [30]:
with engine.begin() as conn:

    conn.execute(text("""
        CREATE OR REPLACE VIEW vw_kpi_transactions AS
        SELECT
            agence_id,
            produit_id,
            annee,
            mois,
            COUNT(*) AS nombre_transactions,
            SUM(montant_eur) AS total_montant_eur,
            AVG(montant_eur) AS moyenne_montant_eur
        FROM transactions
        GROUP BY agence_id, produit_id, annee, mois;
    """))

    conn.execute(text("""
        CREATE OR REPLACE VIEW vw_kpi_taux_defaut AS
        SELECT
            sc.categorie_risque,
            COUNT(*) AS nombre_transactions,
            COUNT(*) FILTER (
                WHERE t.statut IN ('Rejete', 'Refusé', 'Défaut')
            ) AS nombre_defauts,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE t.statut IN ('Rejete', 'Refusé', 'Défaut')
                ) / NULLIF(COUNT(*), 0),
                2
            ) AS taux_defaut_pourcentage
        FROM transactions t
        JOIN clients c ON t.client_id = c.client_id
        JOIN segments_client sc ON c.segment_id = sc.segment_id
        GROUP BY sc.categorie_risque;
    """))

    conn.execute(text("""
        CREATE OR REPLACE VIEW vw_kpi_agences AS
        SELECT
            agence_id,
            COUNT(*) AS nombre_transactions,
            SUM(montant_eur) AS montant_total,
            AVG(montant_eur) AS montant_moyen,
            COUNT(DISTINCT client_id) AS nombre_clients_uniques
        FROM transactions
        GROUP BY agence_id;
    """))

    conn.execute(text("""
        CREATE OR REPLACE VIEW vw_kpi_produits AS
        SELECT
            produit_id,
            COUNT(*) AS nombre_transactions,
            SUM(montant_eur) AS montant_total,
            AVG(montant_eur) AS montant_moyen
        FROM transactions
        GROUP BY produit_id;
    """))

print("✅ Views created successfully")

✅ Views created successfully


In [31]:
df = pd.read_sql("SELECT * FROM vw_kpi_transactions", engine)
df.head()

,agence_id,produit_id,annee,mois,nombre_transactions,total_montant_eur,moyenne_montant_eur
0,1,4,2022,6,1,774.55,774.550000
1,1,4,2023,1,1,465.68,465.680000
2,8,8,2022,6,1,-4329.79,-4329.790000
3,1,7,2024,1,3,-5521.13,-1840.376667
4,4,1,2023,4,1,458.60,458.600000
